# M6 — TabNet Employee Resignation Model

**Owner:** Member 6  
**Primary selection metric:** development out-of-fold average precision (PR-AUC)  
**Formal holdout status:** prohibited in this notebook

This notebook runs the frozen TabNet experiment contract. Category vocabularies, numerical imputation, and numerical scaling are learned inside each outer-training fold. The outer validation fold is used only for out-of-fold prediction; TabNet early stopping uses a stratified inner split created by the adapter.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path("../..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation.metrics import calculate_binary_metrics
from src.models.runner import run_5fold_cv
from src.models.tabnet_model import TabNetAdapter, load_tabnet_config

CONFIG = load_tabnet_config(PROJECT_ROOT / "configs" / "models" / "tabnet.yaml")
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "Extended_Employee_Performance_and_Productivity_Data.csv"
REGISTRY_PATH = PROJECT_ROOT / "data" / "processed" / "split_registry.csv"
OUTPUT_DIR = PROJECT_ROOT / "results" / "models" / "tabnet"
FEATURES = CONFIG["preprocessing"]["categorical_features"] + CONFIG["preprocessing"]["numerical_features"]
TARGET = "Resigned"
ID_COLUMN = "Employee_ID"

print(f"Project root: {PROJECT_ROOT}")
print(f"TabNet candidates: {[item['id'] for item in CONFIG['candidates']]}")

Project root: C:\Users\Kawya\Documents\AIML Project
TabNet candidates: ['T01', 'T02', 'T03', 'T04', 'T05', 'T06', 'T07', 'T08']


## Final TabNet selection summary and additional-seed sensitivity

The frozen selection rule ranks the eight candidates by pooled development OOF average precision. Runtime, fold stability, calibration, and threshold behaviour are secondary evidence. After selecting the winner, two additional deterministic seeds are evaluated with the same development rows and five outer folds. The formal holdout remains prohibited.

In [2]:
from copy import deepcopy

RUN_SEED_SENSITIVITY = False
SELECTED_CANDIDATE = "T01"
BASE_SEED = 42
ADDITIONAL_SEEDS = [7, 2026]
SELECTION_DIR = OUTPUT_DIR / "selection"
SEED_DIR = OUTPUT_DIR / "seed_sensitivity"

raw = pd.read_csv(RAW_PATH)
registry = pd.read_csv(REGISTRY_PATH)
development_registry = registry.loc[
    registry["partition"] == "development", [ID_COLUMN, "cv_fold"]
].copy()
holdout_ids = set(registry.loc[registry["partition"] == "holdout", ID_COLUMN])
development = raw.merge(
    development_registry, on=ID_COLUMN, how="inner", validate="one_to_one"
)
assert len(development) == 80_000
assert set(development[ID_COLUMN]).isdisjoint(holdout_ids)
assert sorted(development["cv_fold"].unique().tolist()) == [1, 2, 3, 4, 5]

candidate_rows = []
for candidate_path in sorted(OUTPUT_DIR.glob("t??")):
    summary_path = candidate_path / "summary_metrics.json"
    if summary_path.exists():
        summary = json.loads(summary_path.read_text(encoding="utf-8"))
        candidate_rows.append({"candidate": candidate_path.name.upper(), **summary})

candidate_comparison = pd.DataFrame(candidate_rows).sort_values(
    "pooled_average_precision", ascending=False
).reset_index(drop=True)
assert len(candidate_comparison) == 8
assert candidate_comparison.loc[0, "candidate"] == SELECTED_CANDIDATE
SELECTION_DIR.mkdir(parents=True, exist_ok=True)
candidate_comparison.to_csv(SELECTION_DIR / "candidate_comparison.csv", index=False)
display(candidate_comparison[[
    "candidate", "pooled_average_precision", "cv_mean_average_precision",
    "cv_std_average_precision", "pooled_roc_auc", "pooled_brier_score",
    "total_fit_time_seconds",
]])

if RUN_SEED_SENSITIVITY:
    SEED_DIR.mkdir(parents=True, exist_ok=True)
    for seed in ADDITIONAL_SEEDS:
        seed_config = deepcopy(CONFIG)
        seed_config["training"]["random_seed"] = seed
        seed_config["fixed_params"]["seed"] = seed
        seed_config["reproducibility"]["seed"] = seed
        seed_result = run_5fold_cv(
            adapter_factory=lambda seed_config=seed_config: TabNetAdapter(
                config=seed_config, candidate_id=SELECTED_CANDIDATE
            ),
            data=development.drop(columns=["cv_fold"]),
            split_registry=registry,
            feature_cols=FEATURES,
            target_col=TARGET,
            expected_model_name="TabNet",
        )
        seed_output_dir = SEED_DIR / f"{SELECTED_CANDIDATE.lower()}_seed_{seed}"
        seed_output_dir.mkdir(parents=True, exist_ok=True)
        seed_result.predictions.to_csv(seed_output_dir / "oof_predictions.csv", index=False)
        seed_result.fold_metrics.to_csv(seed_output_dir / "fold_metrics.csv", index=False)
        seeded_summary = {"random_seed": seed, **seed_result.summary_metrics}
        (seed_output_dir / "summary_metrics.json").write_text(
            json.dumps(seeded_summary, indent=2), encoding="utf-8"
        )

seed_rows = []
base_summary = json.loads(
    (OUTPUT_DIR / SELECTED_CANDIDATE.lower() / "summary_metrics.json").read_text(encoding="utf-8")
)
seed_rows.append({"random_seed": BASE_SEED, **base_summary})
for seed in ADDITIONAL_SEEDS:
    seeded_path = SEED_DIR / f"{SELECTED_CANDIDATE.lower()}_seed_{seed}" / "summary_metrics.json"
    if seeded_path.exists():
        seed_rows.append(json.loads(seeded_path.read_text(encoding="utf-8")))

seed_sensitivity = pd.DataFrame(seed_rows).sort_values("random_seed").reset_index(drop=True)
if len(seed_sensitivity) == 1 + len(ADDITIONAL_SEEDS):
    seed_sensitivity.to_csv(SELECTION_DIR / "seed_sensitivity.csv", index=False)
    ap_values = seed_sensitivity["pooled_average_precision"].astype(float)
    selection_summary = {
        "selection_metric": "pooled_average_precision",
        "selected_candidate": SELECTED_CANDIDATE,
        "primary_comparison_seed": BASE_SEED,
        "additional_sensitivity_seeds": ADDITIONAL_SEEDS,
        "seed_count": int(len(seed_sensitivity)),
        "mean_pooled_average_precision_across_seeds": float(ap_values.mean()),
        "std_pooled_average_precision_across_seeds": float(ap_values.std(ddof=0)),
        "min_pooled_average_precision_across_seeds": float(ap_values.min()),
        "max_pooled_average_precision_across_seeds": float(ap_values.max()),
        "formal_holdout_used": False,
    }
    (SELECTION_DIR / "selection_summary.json").write_text(
        json.dumps(selection_summary, indent=2), encoding="utf-8"
    )
    display(seed_sensitivity[[
        "random_seed", "pooled_average_precision", "cv_mean_average_precision",
        "cv_std_average_precision", "pooled_roc_auc", "pooled_brier_score",
        "total_fit_time_seconds",
    ]])
    display(pd.Series(selection_summary, name="selected_tabnet"))
else:
    print("Seed sensitivity is incomplete; enable RUN_SEED_SENSITIVITY to generate missing runs.")

,candidate,pooled_average_precision,cv_mean_average_precision,cv_std_average_precision,pooled_roc_auc,pooled_brier_score,total_fit_time_seconds
0,T01,0.101706,0.101955,0.001801,0.504556,0.090267,504.668862
1,T02,0.100757,0.102074,0.004120,0.500338,0.253767,586.403368
2,T05,0.100672,0.100980,0.001187,0.501184,0.091430,234.304518
3,T08,0.100571,0.101575,0.001304,0.502327,0.251642,2107.993343
4,T04,0.100564,0.100444,0.002337,0.499674,0.248368,848.018646
5,T06,0.100555,0.100486,0.001026,0.503286,0.261512,780.774142
6,T07,0.100503,0.100238,0.001818,0.500173,0.090800,3440.982471
7,T03,0.100487,0.100702,0.001586,0.501749,0.090404,511.405817


,random_seed,pooled_average_precision,cv_mean_average_precision,cv_std_average_precision,pooled_roc_auc,pooled_brier_score,total_fit_time_seconds
0,7,0.099065,0.099840,0.003320,0.496543,0.090455,1768.516028
1,42,0.101706,0.101955,0.001801,0.504556,0.090267,504.668862
2,2026,0.101108,0.101471,0.001221,0.502297,0.090325,772.750394


selection_metric                              pooled_average_precision
selected_candidate                                                 T01
primary_comparison_seed                                             42
additional_sensitivity_seeds                                 [7, 2026]
seed_count                                                           3
mean_pooled_average_precision_across_seeds                    0.100626
std_pooled_average_precision_across_seeds                     0.001131
min_pooled_average_precision_across_seeds                     0.099065
max_pooled_average_precision_across_seeds                     0.101706
formal_holdout_used                                              False
Name: selected_tabnet, dtype: object

## 1. Load only the frozen development partition

The registry—not a new random split—defines the 80,000 development rows and five shared folds. This cell explicitly checks that no formal-holdout employee enters the modelling dataframe.

In [3]:
raw = pd.read_csv(RAW_PATH)
registry = pd.read_csv(REGISTRY_PATH)

development_registry = registry.loc[registry["partition"] == "development", [ID_COLUMN, "cv_fold"]].copy()
holdout_ids = set(registry.loc[registry["partition"] == "holdout", ID_COLUMN])
development = raw.merge(development_registry, on=ID_COLUMN, how="inner", validate="one_to_one")

print(f"Raw shape: {raw.shape}")
print(f"Development shape after registry filtering: {development.shape}")
assert len(development) == 80_000
assert set(development[ID_COLUMN]).isdisjoint(holdout_ids)
assert sorted(development["cv_fold"].unique().tolist()) == [1, 2, 3, 4, 5]
assert set(FEATURES).issubset(development.columns)
assert ID_COLUMN not in FEATURES and "Hire_Date" not in FEATURES and TARGET not in FEATURES

target_summary = development[TARGET].value_counts(dropna=False).rename_axis(TARGET).to_frame("count")
target_summary["proportion"] = target_summary["count"] / len(development)
display(target_summary)

Raw shape: (100000, 20)
Development shape after registry filtering: (80000, 21)


,count,proportion
Resigned,,
False,71992,0.8999
True,8008,0.1001


## 2. Dependency and save/reload smoke test

Run this only after the pinned PyTorch and `pytorch-tabnet` packages are installed. It uses 5,000 rows from the training side of shared fold 1 and a small untouched subset of fold 1 for pipeline verification. These smoke metrics are diagnostic only and must not be reported as formal model results. The save/reload assertion verifies that serialized preprocessing and model parameters reproduce probabilities.

In [4]:
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    smoke_config = CONFIG["smoke_test"]
    outer_training = development.loc[development["cv_fold"] != 1].reset_index(drop=True)
    outer_validation = development.loc[development["cv_fold"] == 1].reset_index(drop=True)

    smoke_train_idx, _ = train_test_split(
        np.arange(len(outer_training)),
        train_size=int(smoke_config["sample_rows"]),
        random_state=int(CONFIG["training"]["random_seed"]),
        stratify=outer_training[TARGET],
    )
    smoke_valid_idx, _ = train_test_split(
        np.arange(len(outer_validation)),
        train_size=1_000,
        random_state=int(CONFIG["training"]["random_seed"]),
        stratify=outer_validation[TARGET],
    )
    smoke_train = outer_training.iloc[smoke_train_idx]
    smoke_valid = outer_validation.iloc[smoke_valid_idx]
    print(f"Smoke training shape before/after sampling: {outer_training.shape} -> {smoke_train.shape}")
    print(f"Smoke validation shape before/after sampling: {outer_validation.shape} -> {smoke_valid.shape}")

    smoke_adapter = TabNetAdapter(
        config=CONFIG,
        candidate_id=smoke_config["candidate_id"],
        max_epochs_override=int(smoke_config["max_epochs"]),
        patience_override=int(smoke_config["patience"]),
    )
    smoke_adapter.fit(smoke_train[FEATURES], smoke_train[TARGET])
    smoke_probability = smoke_adapter.predict_proba(smoke_valid[FEATURES])
    smoke_metrics = calculate_binary_metrics(smoke_valid[TARGET].astype(int), smoke_probability)
    display(pd.Series(smoke_metrics, name="diagnostic_smoke_metric"))

    smoke_artifact_dir = OUTPUT_DIR / "smoke_artifact"
    smoke_adapter.save(smoke_artifact_dir)
    reloaded_adapter = TabNetAdapter.load(smoke_artifact_dir)
    reloaded_probability = reloaded_adapter.predict_proba(smoke_valid[FEATURES])
    np.testing.assert_allclose(smoke_probability, reloaded_probability, rtol=1e-6, atol=1e-7)
    print("Smoke fit, prediction, serialization, and reload checks passed.")
else:
    print("Smoke training is disabled. Set RUN_SMOKE_TEST=True only after dependency approval.")

Smoke training is disabled. Set RUN_SMOKE_TEST=True only after dependency approval.


## 3. Formal candidate run on the five shared development folds

Run one declared candidate at a time and retain every OOF result. Candidate selection happens only after comparing all completed development OOF results by pooled average precision, with runtime and stability as secondary evidence. Do not open, score, or predict the formal holdout here.

In [5]:
RUN_FORMAL_CV = False
CANDIDATE_TO_RUN = "T08"

if RUN_FORMAL_CV:
    declared_ids = {item["id"] for item in CONFIG["candidates"]}
    assert CANDIDATE_TO_RUN in declared_ids
    result = run_5fold_cv(
        adapter_factory=lambda: TabNetAdapter(config=CONFIG, candidate_id=CANDIDATE_TO_RUN),
        data=development.drop(columns=["cv_fold"]),
        split_registry=registry,
        feature_cols=FEATURES,
        target_col=TARGET,
        expected_model_name="TabNet",
    )
    candidate_dir = OUTPUT_DIR / CANDIDATE_TO_RUN.lower()
    candidate_dir.mkdir(parents=True, exist_ok=True)
    result.predictions.to_csv(candidate_dir / "oof_predictions.csv", index=False)
    result.fold_metrics.to_csv(candidate_dir / "fold_metrics.csv", index=False)
    (candidate_dir / "summary_metrics.json").write_text(
        json.dumps(result.summary_metrics, indent=2), encoding="utf-8"
    )
    display(result.fold_metrics)
    display(pd.Series(result.summary_metrics, name=CANDIDATE_TO_RUN))
else:
    print("Formal five-fold TabNet training is disabled until the smoke and reload checks pass.")

Formal five-fold TabNet training is disabled until the smoke and reload checks pass.


## Git checkpoint

Commit the TabNet configuration, adapter, tests, and this clean unexecuted scaffold after the core tests pass. Push the trained-result commit only after the dependency smoke test, save/reload check, five-fold OOF validation, and notebook Restart & Run All all pass. Never commit the raw employee dataset, local virtual environment, notebook checkpoints, or temporary smoke artifacts.